In [16]:
import pandas as pd 
import polars as pl
import sys 
import os 
import numpy as np 
sys.path.append("..")

from settings import NB_TRANSACTIONS_PER_MONTH, PROJECT_PATH, REGRESSION_TARGET,CLASSIFICATION_TARGET

In [17]:
transactions = pl.read_parquet(
    os.path.join(PROJECT_PATH, "real_estate_transactions_engineered.parquet")
)

As a slightly more sophisticated benchmark, we can extend the idea of using the target mean by computing the city-level average price per m² for each month instead. This information was already computed during feature engineering and is stored in the column avg_price_per_m2_previous_month.

In [18]:
transactions["avg_price_per_m2_previous_month"]

avg_price_per_m2_previous_month
f64
1241.194799
1241.194799
1241.194799
1241.194799
1241.194799
…
3853.821448
3853.821448
3711.043574


La "prediction" du "modèle" devient alors tout simplement la valeur de cette colonne

In [19]:
X = transactions.drop([REGRESSION_TARGET, CLASSIFICATION_TARGET])
y_pred = transactions["avg_price_per_m2_previous_month"]
y_regression = transactions[REGRESSION_TARGET]

Nous avons désormais des "predictions" ainsi que notre target. On peut alors calculer toutes nos métriques de performances comme nous le souhaitons ! 

In [20]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [22]:

rmse = np.sqrt(mean_squared_error(y_pred, y_regression))
print("RMSE : ", rmse)
mae = mean_absolute_error(y_pred, y_regression)
print("MAE : ", mae)
r2 = r2_score(y_pred, y_regression)
print("R2 : ", r2)

RMSE :  371474.18203455745
MAE :  242045.18741309742
R2 :  -13640.792490333895


A negative R²! There is clearly hope that an ML model can do better. This result is also interesting because it tells us that reliably estimating a transaction price based solely on a global average is not realistic.

Notice that we did not perform a train-test split here. This would indeed be unnecessary, because we are not using a statistical model but a deterministic rule that is not subject to overfitting or underfitting. 